In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from sklearn.metrics import accuracy_score, classification_report
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(test_ds.classes)  

model_path = 'models/VGG16.keras'
try:
    modelk = tf.keras.models.load_model(model_path)
    print("Model loaded successfully!")
    modelk.summary()
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the path to your .keras file is correct and TensorFlow is installed.")


def inference(model1, loader):
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs_batch, labels_batch in loader:
    
            for i in range(len(imgs_batch)):
                img_single = imgs_batch[i].unsqueeze(0).to(device) 
                label_single = labels_batch[i].item() 
                keras_input = img_single.permute(0, 2, 3, 1).cpu().numpy()
                predictions_keras = model1.predict(keras_input, verbose=0)
                out1 = torch.from_numpy(predictions_keras).to(device)
                prob1 = F.softmax(out1, dim=1)
                pred_single = prob1.argmax(dim=1)
                y_true.append(label_single)
                y_pred.append(pred_single)

    return y_true, y_pred


y_true, y_pred = inference(
    modelk,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))